# Create a Structured Meal & Grocery Planner with CrewAI

<img src='../assets/flow.png' width='800' />

`For this lab, we need SERPER_API_KEY and OPENAI_API_KEY`

In [24]:
%%capture

from pydantic import BaseModel, Field
from typing import List, Dict, Optional
from IPython.display import display, JSON, Markdown
from datetime import datetime
import os

import litellm
litellm.ssl_verify = False

import os
from crewai_tools import SerperDevTool
from crewai import LLM, Agent, Task, Crew, Process

# Initialize LLM
llm = LLM(model="openai/gpt-4.1-nano")

from dotenv import load_dotenv
load_dotenv()

from leftover import LeftoversCrew

## Create Data Structures

### MealPlan

In [2]:
class MealPlan(BaseModel):
    """Simple meal plan"""
    meal_name: str = Field(description="Name of the meal")
    difficulty_level: str = Field(description="'Easy', 'Medium', 'Hard'") # How challenging it is to cook
    servings: int = Field(description="Number of people it serves") # How many people will be served from our meal
    researched_ingredients: List[str] = Field(description="Ingredients found through research")

In [3]:
# # Its just for example. How to initilize MealPlan
# sample_meal = MealPlan(
#     meal_name="Chicken Stir Fry",
#     difficulty_level="Easy",
#     servings=4,
#     researched_ingredients=["chicken breast", "broccoli", "bell peppers", "garlic", "soy sauce", "rice"]
# )

### Grocery Shopping Data

In [4]:
class GroceryItem(BaseModel):
    """Individual grocery item"""
    name: str = Field(description="Name of the grocery item")
    quantity: str = Field(description="Quantity needed (for example, '2 lbs', '1 gallon')")
    estimated_price: str = Field(description="Estimated price (for example, '$3-5')")
    category: str = Field(description="Store section (for example, 'Produce', 'Dairy', 'Meat')") # In which store section, this grocery item will be found

In [5]:
# # Its just for example. How to initilize GroceryItem
# sample_item = GroceryItem(
#     name="Chicken Breast",
#     quantity="2 lbs",
#     estimated_price="$8-12",
#     category="Meat"
# )

### ShoppingCategory

In [6]:
class ShoppingCategory(BaseModel):
    """Store section with items"""
    section_name: str = Field(description="Store section (for example, 'Produce', 'Dairy')")
    items: List[GroceryItem] = Field(description="Items in this section") # In one section: which GroceryItems need to purchase
    estimated_total: str = Field(description="Estimated cost for this section")

In [7]:
# # Its just for example. How to initilize ShoppingCategory
# sample_section = ShoppingCategory(
#     section_name="Produce",
#     items=[
#         GroceryItem(name="Bell Peppers", quantity="3 pieces", estimated_price="$3-4", category="Produce"),
#         GroceryItem(name="Onions", quantity="2 lbs", estimated_price="$2-3", category="Produce")
#     ],
#     estimated_total="$5-7"
# )


### GroceryShoppingPlan

In [8]:
class GroceryShoppingPlan(BaseModel):
    """Complete simplified shopping plan"""
    total_budget: str = Field(description="Total planned budget") # The overall spending limit for the shopping - Its our budget for Meal.
    meal_plans: List[MealPlan] = Field(description="Planned meals")
    shopping_sections: List[ShoppingCategory] = Field(description="Organized by store sections") # Organized categories for store navigation 
    shopping_tips: List[str] = Field(description="Money-saving and efficiency tips")

In [9]:
# # Its just for example. How to initilize GroceryShoppingPlan
# weekly_shopping_plan = GroceryShoppingPlan(
#     total_budget="$40-50",
#     meal_plans=[breakfast_meal, lunch_meal, dinner_meal],  # List of MealPlan objects
#     shopping_sections=[dairy_section, meat_section, pantry_section, produce_section],  # List of ShoppingCategory objects
#     shopping_tips=[...]
# )

## Create Our AI Agent Workflow

### Create Meal Planning Agent and Task

In [ ]:
meal_planner = Agent(
    role="Meal Planner & Recipe Researcher",
    goal="Search for optimal recipes and create detailed meal plans",
    backstory="A skilled meal planner who researches the best recipes online, considering dietary needs, cooking skill levels, and budget constraints.",
    tools=[SerperDevTool()],
    llm=llm,
    verbose=False,
    # max_iter=2, # limits to 2 tool calls max
    # max_rpm=10, # max requests per minute to LLM
)

In [ ]:
meal_planning_task = Task(
    description=(
        "Search for the best '{meal_name}' recipe for {servings} people within a {budget} budget. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill level: {cooking_skill}. "
        "Find recipes that match the skill level and provide complete ingredient lists with quantities."
    ),
    expected_output="A detailed meal plan with researched ingredients, quantities, and cooking instructions appropriate for the skill level.",
    agent=meal_planner,
    output_pydantic=MealPlan, # for pydantic output
    # output_json=MealPlan, For json output
    output_file="meals.json"
)

#### Test MEAL PLANNER Crew

In [ ]:
# meal_planner_crew = Crew(
#     agents=[meal_planner],
#     tasks=[meal_planning_task],
#     process=Process.sequential,  # Ensures tasks are executed in order
#     verbose=True
# )

# meal_planner_result = meal_planner_crew.kickoff(
#     inputs={
#         "meal_name": "Chicken Stir Fry",
#         "servings": 4,
#         "budget": "$25",                           
#         "dietary_restrictions": ["no nuts"], # e.g; Gluten Free etc...
#         "cooking_skill": "beginner" # Whats skill level of person, who wanna cook
#     }
# )
# print("✅ Single meal planning completed!")
# print("📋 Single Meal Results:")
# print(meal_planner_result)

### Create Shopping Agent and Task

In [13]:
shopping_organizer = Agent(
    role="Shopping Organizer", 
    goal="Organize grocery lists by store sections efficiently",
    backstory="An experienced shopper who knows how to organize lists for quick store trips and considers dietary restrictions.",
    tools=[],
    llm=llm,
    verbose=False
)

In [ ]:
shopping_task = Task(
    description=(
        "Organize the ingredients from the '{meal_name}' meal plan into a grocery shopping list. "
        "Group items by store sections and estimate quantities for {servings} people. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill: {cooking_skill}. "
        "Stay within budget: {budget}."
    ),
    expected_output="An organized shopping list grouped by store sections with quantities and prices.",
    agent=shopping_organizer,
    context=[meal_planning_task], # to access the meal planner's output
    # process=Process.sequential, # default value
    output_pydantic=GroceryShoppingPlan,
    output_file="shopping_list.json"
)

#### `Process.sequential` vs `context`

**`Process.sequential`** — controls **when** tasks run (one by one, in listed order).

**CrewAI (automatic behavior)** — when running sequentially, every task automatically receives the outputs of **all previous tasks**.

**`context=[...]`** — lets you **override** the automatic behavior. Instead of getting all previous outputs, the task only gets outputs from the tasks you explicitly list.

---

**Think of it this way:**
```
Without context:   task 4 gets → task1 + task2 + task3 outputs  (everything)
With context=[task1]:  task 4 gets → only task1 output           (selective)
```

#### Test MEAL PLANNER and SHOPPING ORGANIZER Crew

In [ ]:
# two_agent_grocery_crew = Crew(
#     agents=[meal_planner, shopping_organizer],  # Both agents
#     tasks=[meal_planning_task, shopping_task],   # Both tasks
#     process=Process.sequential,
#     verbose=True
# )

# # Run the complete crew (this will do BOTH meal planning AND shopping)
# shopping_result = two_agent_grocery_crew.kickoff(
#     inputs={
#         "meal_name": "Chicken Stir Fry",
#         "servings": 4,
#         "budget": "$25",                           
#         "dietary_restrictions": ["no nuts"],      
#         "cooking_skill": "beginner"               
#     }
# )

# # Print the shopping results
# print("✅ Complete meal planning + shopping completed!")
# print("📋 Shopping Results:")
# print(shopping_result)

### Create Budget Advisor Agent and Task

`Add Financial Intelligence`

In [18]:
budget_advisor = Agent(
    role="Budget Advisor",
    goal="Provide cost estimates and money-saving tips",
    backstory="A budget-conscious shopper who helps families save money on groceries while respecting dietary needs.",
    # tools=[SerperDevTool()],
    llm=llm,
    verbose=False
)

In [19]:
budget_task = Task(
    description=(
        "Analyze the shopping plan for '{meal_name}' serving {servings} people. "
        "Ensure total cost stays within {budget}. Consider dietary restrictions: {dietary_restrictions}. "
        "Provide practical money-saving tips and alternative ingredients if needed to meet budget."
    ),
    expected_output="A complete shopping guide with detailed prices, budget analysis, and money-saving tips.",
    agent=budget_advisor,
    context=[meal_planning_task, shopping_task],
    output_file="shopping_guide.md"
)

### Create Food Leftover Agent and Task - Using YAML with CrewAI

#### CrewBase + YAML — Quick Reference

**The Problem (without YAML):**
You write agents and tasks directly in Python — role, goal, backstory all hardcoded in the notebook. Gets messy as the crew grows.

---

**The Solution: `@CrewBase` + YAML**

Instead of this (old way — rest of this notebook):
```python
Agent(role="...", goal="...", backstory="...")
Task(description="...", expected_output="...")
```

You do this (YAML way — `leftover.py`):
```python
@CrewBase
class LeftoversCrew:
    @agent
    def leftover_manager(self):
        return Agent(config=self.agents_config["leftover_manager"], llm=self.llm)

    @task
    def leftover_task(self):
        return Task(config=self.tasks_config["leftover_task"])
```

And the actual content (role, goal, backstory, description) lives in `config/agents.yaml` and `config/tasks.yaml`.

---

**What `@CrewBase` does automatically:**
- Reads `config/agents.yaml` → loads into `self.agents_config`
- Reads `config/tasks.yaml` → loads into `self.tasks_config`
- No manual file loading needed — just reference by key name

---

**Why use YAML?**

| Python (inline) | YAML |
|---|---|
| Config mixed with code | Config separate from code |
| Hard to edit without touching logic | Non-developers can edit prompts |
| One big file | Clean, organized files |

**Short version:** YAML moves agent/task descriptions out of Python into config files. `@CrewBase` auto-loads those files so you just reference them by name.

### Why a Separate `.py` File + `config/` Folder?

**Why `leftover.py` instead of defining the class in the notebook?**

`@CrewBase` uses Python's `inspect` module to find the class file path:
```python
Path(inspect.getfile(cls)).parent
```
This works in `.py` files but **fails in Jupyter notebooks** — cells have no real file path.
So the crew class must live in a separate `.py` file.

---

**Why must YAML files be inside a `config/` folder?**

`@CrewBase` looks for YAML files in a `config/` directory **relative to the `.py` file** automatically.
So the folder structure must be:
```
2.project-meal-planner/
├── leftover.py           ← crew class lives here
├── config/
│   ├── agents.yaml       ← agent role, goal, backstory
│   └── tasks.yaml        ← task description, expected_output
└── planner.ipynb         ← imports LeftoversCrew from leftover.py
```

In [ ]:
from leftover import LeftoversCrew

leftovers_cb = LeftoversCrew(llm=llm)
yaml_leftover_manager = leftovers_cb.leftover_manager()
yaml_leftover_task = leftovers_cb.leftover_task()

### Defining Our Summary Agent and Task

In [26]:
summary_agent = Agent(
    role="Report Compiler",
    goal="Compile comprehensive meal planning reports from all team outputs",
    backstory="A skilled coordinator who organizes information from multiple specialists into comprehensive, easy-to-follow reports.",
    tools=[],
    llm=llm,
    verbose=False
)

In [27]:
summary_task = Task(
    description=(
        "Compile a comprehensive meal planning report that includes:\n"
        "1. The complete recipe and cooking instructions from the meal planner\n"
        "2. The organized shopping list with prices from the shopping organizer\n"
        "3. The budget analysis and money-saving tips from the budget advisor\n"
        "4. The leftover management suggestions from the waste reduction specialist\n"
        "Format this as a complete, user-friendly meal planning guide."
    ),
    expected_output="A comprehensive meal planning guide that combines all team outputs into one cohesive report.",
    agent=summary_agent,
    context=[meal_planning_task, shopping_task, budget_task, yaml_leftover_task],
)

## Assembling Our Complete Grocery Planning Team

Next, we bring together all five specialists into one powerful grocery planning crew. This represents our complete end-to-end solution from meal idea to comprehensive shopping guide with zero-waste strategies.

**Our Five-Agent Team:**
1. **Meal Planner**: Researches recipes and creates meal plans
2. **Shopping Organizer**: Transforms meal plans into organized shopping lists  
3. **Budget Advisor**: Adds financial analysis and money-saving strategies
4. **Leftover Manager**: Minimizes food waste by suggesting creative uses for leftover ingredients
5. **Report Compiler**: Consolidates all outputs into one comprehensive, user-friendly guide


In [ ]:
complete_grocery_crew = Crew(
    agents=[
        meal_planner,           
        shopping_organizer,      
        budget_advisor,         
        yaml_leftover_manager,  # YAML-based leftover manager
        summary_agent           # New summary agent
    ],
    tasks=[
        meal_planning_task,     
        shopping_task,          
        budget_task,            
        yaml_leftover_task,    # YAML-based leftover task
        summary_task            # New summary task
    ],
    process=Process.sequential, # Runs tasks in order they are listed
    verbose=True
)